# Aethon KB — Ask anything

Full pipeline per query:
`HyDE rewrite` → `Chroma retrieval` → `LLM rerank` → `Generate answer`

## Setup

In [1]:
import os, json, logging
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
load_dotenv(dotenv_path=os.path.join("..", ".env"))
oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

chroma     = chromadb.PersistentClient(path="./chroma_db")
collection = chroma.get_collection(name="aethon_kb")

EMBED_MODEL = "text-embedding-3-small"   # must match ingest
LLM_MODEL   = "gpt-4o-mini"
TOP_K       = 10    # chunks pulled from Chroma
RERANK_K    = 4     # chunks kept after reranking

print(f"Ready — {collection.count()} chunks in collection")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Ready — 31 chunks in collection


## Pipeline

In [2]:
# ── Structured output schemas ──────────────────────────────────────────────────

RERANK_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "rerank_scores",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "scores": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "index":     {"type": "integer"},
                            "score":     {"type": "integer",
                                         "description": "0-10 relevance to the question."},
                            "reasoning": {"type": "string"}
                        },
                        "required": ["index", "score", "reasoning"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["scores"],
            "additionalProperties": False
        }
    }
}


# ── Step 1: embed ──────────────────────────────────────────────────────────────
def embed(text: str) -> list[float]:
    v = np.array(
        oai.embeddings.create(input=[text], model=EMBED_MODEL).data[0].embedding,
        dtype=np.float32
    )
    return (v / np.linalg.norm(v)).tolist()


# ── Step 2: HyDE rewrite ───────────────────────────────────────────────────────
def hyde(question: str) -> str:
    """Generate a hypothetical answer passage, embed that instead of the question."""
    return oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "Write a short factual passage (3-5 sentences) that directly answers "
             "the question. Write confidently, as if from a company knowledge base. "
             "No hedging."},
            {"role": "user", "content": question},
        ],
        temperature=0,
        max_tokens=200,
    ).choices[0].message.content.strip()


# ── Step 3: retrieve from Chroma ───────────────────────────────────────────────
def retrieve(query_vec: list[float], n: int = TOP_K) -> list[dict]:
    r = collection.query(
        query_embeddings=[query_vec],
        n_results=n,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "index":    i,
            "chunk_id": r["ids"][0][i],
            "topic":    r["metadatas"][0][i]["topic"],
            "filename": r["metadatas"][0][i]["filename"],
            "score":    round(1 - r["distances"][0][i], 4),
            "text":     r["documents"][0][i],
        }
        for i in range(len(r["ids"][0]))
    ]


# ── Step 4: LLM rerank ─────────────────────────────────────────────────────────
def rerank(question: str, candidates: list[dict], top_k: int = RERANK_K) -> list[dict]:
    """Score each candidate 0-10 against the ORIGINAL question, return top_k."""
    listing = "\n\n".join(f"[{c['index']}] {c['text'][:400]}" for c in candidates)
    resp = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "Score each passage 0-10 for how directly it answers the question. "
             "10 = directly answers it. 0 = completely off-topic."},
            {"role": "user", "content": f"QUESTION: {question}\n\n{listing}"},
        ],
        temperature=0,
        max_tokens=1024,
        response_format=RERANK_SCHEMA,
    )
    scores = json.loads(resp.choices[0].message.content)["scores"]
    for s in scores:
        candidates[s["index"]]["llm_score"]     = s["score"]
        candidates[s["index"]]["llm_reasoning"] = s["reasoning"]
    return sorted(candidates, key=lambda x: x.get("llm_score", 0), reverse=True)[:top_k]


# ── Step 5: generate answer ────────────────────────────────────────────────────
def generate(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(c["text"] for c in chunks)
    return oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content":
             "Answer the question using ONLY the provided context. "
             "Be concise and direct. If the answer is not in the context, say so."},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ],
        temperature=0,
        max_tokens=512,
    ).choices[0].message.content.strip()


# ── Main ask() function ────────────────────────────────────────────────────────
def ask(question: str, verbose: bool = False) -> str:
    """
    Full pipeline: HyDE rewrite → retrieve → LLM rerank → generate.
    Set verbose=True to see which chunks were used.
    """
    # 1. HyDE: generate hypothetical answer and embed it
    hypo       = hyde(question)
    query_vec  = embed(hypo)

    # 2. Retrieve top-K from Chroma using the HyDE vector
    candidates = retrieve(query_vec, n=TOP_K)

    # 3. LLM rerank against the ORIGINAL question (not the HyDE)
    top_chunks = rerank(question, candidates, top_k=RERANK_K)

    # 4. Generate answer from reranked chunks
    answer = generate(question, top_chunks)

    if verbose:
        print("── HyDE passage ──────────────────────────────")
        print(hypo)
        print("\n── Chunks used (after rerank) ────────────────")
        for c in top_chunks:
            print(f"  [{c['llm_score']}/10] {c['topic']}  ({c['filename']})")
            print(f"           cosine={c['score']}  reason: {c['llm_reasoning']}")
        print("\n── Answer ────────────────────────────────────")

    return answer


print("ask() ready")

ask() ready


## Try it

In [3]:
print(ask("Who is the CTO of Aethon Dynamics?", verbose=True))

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


── HyDE passage ──────────────────────────────
As of October 2023, the Chief Technology Officer (CTO) of Aethon Dynamics is Dr. Emily Chen. She has been instrumental in leading the company's technological innovations and strategic direction in the field of robotics and automation. Dr. Chen holds a Ph.D. in Robotics and has over 15 years of experience in the industry.

── Chunks used (after rerank) ────────────────
  [10/10] Daniel Osei  (employees.md)
           cosine=0.4798  reason: This passage directly states that Daniel Osei is the Chief Technology Officer of Aethon Dynamics.
  [0/10] Dr. Maya Krishnan  (employees.md)
           cosine=0.6496  reason: This passage provides information about the CEO but does not mention the CTO.
  [0/10] Document Overview  (employees.md)
           cosine=0.556  reason: This document is about the leadership team but does not specify who the CTO is.
  [0/10] Company Structure  (company_overview.md)
           cosine=0.5319  reason: This passage disc

In [4]:
print(ask("What was Aethon's ARR in 2024 and who is their biggest customer?", verbose=True))

── HyDE passage ──────────────────────────────
As of 2024, Aethon's Annual Recurring Revenue (ARR) was reported to be $50 million. The company's largest customer is a major healthcare provider, which utilizes Aethon's automated solutions to enhance operational efficiency and patient care.

── Chunks used (after rerank) ────────────────
  [10/10] Revenue Growth and Breakdown  (sales.md)
           cosine=0.5647  reason: This passage directly provides Aethon's ARR for 2024 as $31.0M and identifies VoltCore as their biggest customer, accounting for 58% of annual recurring revenue.
  [2/10] Overview of Aethon Dynamics  (sales.md)
           cosine=0.5669  reason: This passage mentions Aethon's revenue and sales pipeline but does not provide specific figures for ARR in 2024 or identify any customers.
  [0/10] Company Overview  (company_overview.md)
           cosine=0.5478  reason: This passage gives an overview of Aethon Dynamics but does not address the ARR for 2024 or mention any custome

In [5]:
# Your own question
print(ask("What awards has Aethon won?", verbose=True))

── HyDE passage ──────────────────────────────
Aethon has received several prestigious awards, including the 2019 Best in KLAS award for its automated pharmacy solutions and the 2020 Healthcare Innovation Award for its contributions to improving hospital logistics. Additionally, Aethon was recognized with the 2021 Tech Innovation Award for its advancements in robotics and automation in healthcare settings. These accolades highlight Aethon's commitment to enhancing operational efficiency and patient care through innovative technology.

── Chunks used (after rerank) ────────────────
  [10/10] Awards & Recognition  (achievements.md)
           cosine=0.5534  reason: This passage directly lists several awards that Aethon has won, providing specific details about each award.
  [0/10] Introduction to Aethon Dynamics  (achievements.md)
           cosine=0.612  reason: This passage mentions Aethon's achievements but does not specify any awards.
  [0/10] Company Overview  (company_overview.md)
